In [20]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler,OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split,KFold,GridSearchCV
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_selector,make_column_transformer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore' )

In [2]:
from ISLP import load_data
Carseats = load_data('Carseats')
X,y = Carseats.drop('Sales', axis = 1),Carseats.Sales
X_train, X_test, y_train, y_test = train_test_split(X,y , random_state=26, test_size= 0.2, stratify = Carseats['ShelveLoc'])
Carseats.columns

Index(['Sales', 'CompPrice', 'Income', 'Advertising', 'Population', 'Price',
       'ShelveLoc', 'Age', 'Education', 'Urban', 'US'],
      dtype='object')

In [3]:
y_train.shape

(320,)

In [4]:
Carseats.shape

(400, 11)

In [5]:
Carseats.head()

,Sales,CompPrice,Income,Advertising,Population,Price,ShelveLoc,Age,Education,Urban,US
0,9.50,138,73,11,276,120,Bad,42,17,Yes,Yes
1,11.22,111,48,16,260,83,Good,65,10,Yes,Yes
2,10.06,113,35,10,269,80,Medium,59,12,Yes,Yes
3,7.40,117,100,4,466,97,Medium,55,14,Yes,Yes
4,4.15,141,64,3,340,128,Bad,38,13,Yes,No


In [6]:
Carseats.isna().sum()

Sales          0
CompPrice      0
Income         0
Advertising    0
Population     0
Price          0
ShelveLoc      0
Age            0
Education      0
Urban          0
US             0
dtype: int64

In [22]:
shelves = [['Bad','Medium','Good']]
ord_enc = OrdinalEncoder(categories= shelves)
ohe = OneHotEncoder(sparse_output= False, drop = 'first').set_output(transform = 'pandas')
ct = make_column_transformer((ohe,['Urban','US']),
                             (ord_enc, ['ShelveLoc']),
                             ('passthrough', make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out= False).set_output(transform = 'pandas')
sc = StandardScaler()
mm = MinMaxScaler()
rf = RandomForestRegressor(random_state= 26)
gbm = XGBRegressor(random_state=26)
lgbm = LGBMRegressor(random_state=26,verbose=-1)

pipe1 = Pipeline([('CT',ct),('SC',sc),('RF',rf)])
pipe2 = Pipeline([('CT',ct),('SC',sc),('GBM',gbm)])
pipe3 = Pipeline([('CT',ct),('SC',sc),('LGBM',lgbm)])

kfold = KFold(shuffle = True, random_state=26)

In [8]:
pipe1.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    ['Urban', 'US']),
                                   ('ordinalencoder',
                                    OrdinalEncoder(categories=[['Bad'], ['Medium'],
                                                               ['Good']]),
                                    ['ShelveLoc']),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001A9CA44EBA0>)],
                     verbose_feature_names_out=False)),
  ('SC', StandardScaler()),
  ('RF', RandomForestRegressor(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                           

In [11]:
params={
    'SC':[sc,mm],
    'RF__n_estimators': [25,50,100],
    'RF__max_features': [2,3,4,5,6]
}
gd = GridSearchCV(pipe1, param_grid= params, cv = kfold, scoring = 'r2') 
gd.fit(X_train,y_train)

print('Best parameters:' , gd.best_params_)
print('Best score: ', gd.best_score_)

Best parameters: {'RF__max_features': 5, 'RF__n_estimators': 100, 'SC': MinMaxScaler()}
Best score:  0.6659460492785876


In [12]:
pred = gd.predict(X_test)
r2_score(y_test, pred)

0.646372136523863

In [13]:
pipe2.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    ['Urban', 'US']),
                                   ('ordinalencoder',
                                    OrdinalEncoder(categories=[['Bad', 'Medium',
                                                                'Good']]),
                                    ['ShelveLoc']),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001A9CA4B7230>)],
                     verbose_feature_names_out=False)),
  ('SC', StandardScaler()),
  ('GBM',
   XGBRegressor(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=N

In [15]:
params = { 'SC': [mm,sc],
            'GBM__max_depth':[None, 2,3,4],
           'GBM__learning_rate':np.linspace(0.01, 0.9, 10),
           'GBM__n_estimators': [25,50,100] }
gs=GridSearchCV(pipe2, param_grid= params , cv = kfold, scoring = 'r2')
gs.fit(X_train, y_train)
print('Best parameters:' , gs.best_params_)
print('Best score: ', gs.best_score_)        

Best parameters: {'GBM__learning_rate': np.float64(0.20777777777777778), 'GBM__max_depth': 2, 'GBM__n_estimators': 100, 'SC': MinMaxScaler()}
Best score:  0.7564564206569815


In [16]:
pred1 = gs.predict(X_test)
r2_score(y_test, pred1)

0.786994846981635

In [17]:
pipe2.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    ['Urban', 'US']),
                                   ('ordinalencoder',
                                    OrdinalEncoder(categories=[['Bad', 'Medium',
                                                                'Good']]),
                                    ['ShelveLoc']),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001A9CA4B7230>)],
                     verbose_feature_names_out=False)),
  ('SC', StandardScaler()),
  ('GBM',
   XGBRegressor(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=N

In [23]:
params = { 'LGBM__max_depth': [None, 2, 3, 4],
    'LGBM__learning_rate': np.linspace(0.01, 0.9, 10),
    'LGBM__n_estimators': [25, 50, 100] }
gs=GridSearchCV(pipe3, param_grid= params , cv = kfold, scoring = 'r2')
gs.fit(X_train, y_train)
print('Best parameters:' , gs.best_params_)
print('Best score: ', gs.best_score_)        

Best parameters: {'LGBM__learning_rate': np.float64(0.30666666666666664), 'LGBM__max_depth': 2, 'LGBM__n_estimators': 50}
Best score:  0.7490567495684224


In [25]:
pred = gs.predict(X_test)
r2_score(y_test, pred)

0.7997925183742228

In [24]:
from sklearn.model_selection import RandomizedSearchCV
rgcv = RandomizedSearchCV(pipe3, param_distributions = params, n_iter = 20, cv = kfold, scoring = 'r2', n_jobs = -1)
rgcv.fit(X_train, y_train)
print('Best parameters:' , gs.best_params_)
print('Best score: ', gs.best_score_)    
bm_gbm = rgcv.predict(X_test)
r2_score(y_test, bm_gbm)

Best parameters: {'LGBM__learning_rate': np.float64(0.30666666666666664), 'LGBM__max_depth': 2, 'LGBM__n_estimators': 50}
Best score:  0.7490567495684224


0.785711367398375